<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SQL_Advanced_Joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Advanced: Joins
Hi! Welcome to another part of the course.

By now, you’ve already built a solid foundation in SQL. In this section, we’ll go a bit deeper and focus on gaining **more precise control over JOIN operations**.

If JOINs feel a little fuzzy, don’t worry — we’ll quickly review them before moving on to more advanced usage.

Before we start, let’s take a look at the data we’ll be working with.  
In this notebook, you’ll explore three new tables:

- `student`
- `room`
- `equipment`

These tables will help us practice different JOIN techniques in a more realistic scenario.

Ready? Let’s begin.


## **SQL Environment Setup (do not edit)**

In [22]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [23]:
# @title

DB_FILE = 'class_joins.duckdb'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = duckdb.connect(DB_FILE)

conn.execute(r'''
-- Drop tables if they already exist
DROP TABLE IF EXISTS student;
DROP TABLE IF EXISTS equipment;
DROP TABLE IF EXISTS room;

-- Create tables
CREATE TABLE room (
    id INTEGER PRIMARY KEY,
    room_number INTEGER,
    beds INTEGER,
    floor INTEGER
);

CREATE TABLE student (
    id INTEGER PRIMARY KEY,
    name VARCHAR,
    room_id INTEGER
);

CREATE TABLE equipment (
    id INTEGER PRIMARY KEY,
    name VARCHAR,
    room_id INTEGER
);

-- Insert data into room
INSERT INTO room (id, room_number, beds, floor) VALUES
(1, 101, 2, 1),
(2, 102, 2, 1),
(3, 103, 3, 1),
(4, 104, 3, 1),
(5, 201, 1, 2),
(6, 202, 2, 2),
(7, 203, 3, 2),
(8, 204, 3, 2),
(9, 205, 4, 2),
(10, 301, 4, 3),
(11, 302, 1, 3),
(12, 303, 2, 3),
(13, 401, 3, 4),
(14, 402, 1, 4),
(15, 403, 1, 4);

-- Insert data into student
INSERT INTO student (id, name, room_id) VALUES
(1, 'Jack Pearson', 8),
(2, 'Charlie Black', NULL),
(3, 'Ethan Wright', 15),
(4, 'Mary Benett', NULL),
(5, 'Brian Saunders', 8),
(6, 'Ella Watson', 8),
(7, 'Jacob Chapman', NULL),
(8, 'Charlotte Wood', 1),
(9, 'Emily Lane', 1),
(10, 'Freya Hart', 10),
(11, 'Megan Mcdonald', 10),
(12, 'Noah Rose', 5),
(13, 'Oscar Walls', 10),
(14, 'Luke Wild', 11),
(15, 'Benjamin Slade', 10);

-- Insert data into equipment
INSERT INTO equipment (id, name, room_id) VALUES
(1, 'kettle', 4),
(2, 'fridge', 5),
(3, 'tv', 8),
(4, 'tv', NULL),
(5, 'kettle', 7),
(6, 'radio', 7),
(7, 'computer', 7),
(8, 'toaster', 1),
(9, 'toaster', 1),
(10, 'microwave', NULL),
(11, 'kettle', NULL),
(12, 'kettle', 2),
(13, 'tv', 3),
(14, 'microwave', 9),
(15, 'computer', 10);
''')
print(f"Duckdb database ready ✅ ({DB_FILE})")


Duckdb database ready ✅ (class_joins.duckdb)


## Get to know the table `room`

Alright! Let’s start with the table `room`.

It contains four columns:

- `id` – the unique identifier of the room in the table.
- `room_number` – the number of the room, typically displayed on the dormitory door.
- `beds` – the number of beds available in the room.
- `floor` – the floor where the room is located in the dormitory.

Simple enough, this table describes the physical rooms in the building.

---

## Get to know the table `student`

Next, let’s look at the table `student`.

This table contains three columns:

- `id` – the unique identifier of each student.
- `name` – the full name of the student.
- `room_id` – the id of the room where the student lives.

The column `room_id` refers to the `id` column in the `room` table.

If you look closely at the data, you’ll notice something interesting:  
**some students don’t have a room assigned yet**.

Keep this detail in mind. It will become important later when we work with JOINs.

---

## Get to know the table `equipment`

Finally, let’s examine the third table: `equipment`.

This table is similar in structure to `student` and contains three columns:

- `id` – the identifier of the piece of equipment.
- `name` – the name of the equipment item.
- `room_id` – the id of the room where the equipment is located.

Again, you may notice that **some pieces of equipment are not assigned to any room**.  
These items are simply stored in the warehouse.

This detail will also come in handy when we start exploring different types of JOINs.


## JOIN revised

Do you remember how we joined two tables earlier in the course?  
Let’s quickly revisit the example with **people and their cars**:

```sql
SELECT *
FROM person
JOIN car
  ON person.id = car.owner_id;
````

The structure is simple:

1. `JOIN` tells SQL which table we want to combine with the first one.
2. `ON` defines the **condition that connects the rows** from both tables.

In this example, we match rows where:

* `person.id` (from the `person` table)
* equals `car.owner_id` (from the `car` table)

This way, each car is paired with the person who owns it.

In other words, the query connects **cars with their owners**.


## INNER JOIN

Excellent! One important detail to know is that `JOIN` is actually just one type of join.

When you write `JOIN` in SQL, the database automatically interprets it as an **INNER JOIN**. Since this is the most commonly used join, SQL allows you to omit the word `INNER`.

For example, the query we used earlier:

```sql
SELECT *
FROM person
JOIN car
  ON person.id = car.owner_id;
````

is exactly the same as writing:

```sql
SELECT *
FROM person
INNER JOIN car
  ON person.id = car.owner_id;
```

Both queries produce the same result.
The keyword `INNER` simply makes the type of join **explicit**.


In [24]:
# @title Practice 1
base_practice_1 = make_df_validator_nospoilers(
    expected_hash='fad540e06db3af5053e977175e6a82e2333fe7016345bd53c25d5aa089684049',
    required_cols=['ROOM_ID', 'room_number', 'beds', 'floor', 'EQUIPMENT_ID', 'name'],
    expected_rows=12,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_1 = base_practice_1

make_sql_runner(
    conn,
    runner_id="practice_1",
    description_md='### Practice 1\nUse the full name INNER JOIN to join the room and equipment tables, so that each piece of equipment is shown together with its room and other relevant columns. The result should have the following columns:\n\n  - room_id – ID of the room.\n  - room_number.\n  - beds.\n  - floor.\n  - equipment_id – ID of the equipment.\n  - name (of the equipment).\n',
    validator=val_practice_1,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['room', 'equipment']
)


## How INNER JOIN works

If you compare the result of the `INNER JOIN` with the contents of the `equipment` table, you’ll notice that **not all pieces of equipment appear in the result**.

For example, the kettle with id `11` is missing. Do you know why?

`INNER JOIN` (or simply `JOIN`) only returns rows where **a matching value exists in both tables**.

In our case, rows are matched using the condition:

- `room.id`
- `equipment.room_id`

This means the result will only include equipment that **has a room assigned**.  
If a piece of equipment has `NULL` in `room_id`, it cannot match any room — so it is **excluded from the result**.

In other words:

- equipment **with a room** → appears in the result  
- equipment **without a room** → does not appear

Take a look at the illustration below.

<img src="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/advanced_inner_join_example.png" width="60%">

In the diagram:
- **blue rows** represent the result of the `INNER JOIN`
- **pink rows** represent equipment with `NULL` in `room_id`, which cannot match a room and therefore do not appear in the result


## LEFT JOIN explained

Alright — let’s learn a new type of join: **LEFT JOIN**.

A `LEFT JOIN` returns:
- **all rows from the left table** (the first table in the query)
- plus any **matching rows from the right table**

If a row from the left table has **no match** in the right table, it is still included in the result. In that case, the columns from the right table will contain `NULL`.

Let’s look at an example:

```sql
SELECT *
FROM car
LEFT JOIN person
  ON car.owner_id = person.id;
````

In this query:

* `car` is the **left table**
* `person` is the **right table**

So SQL will return **every car**, whether it has an owner or not.

If a car has a matching owner, the owner’s information is included.
If no owner exists, the car still appears in the result — but the columns from the `person` table will contain `NULL`.

Take a look at the illustration below.

<img src="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/advanced_lef_join_example.png" width="60%">

In the diagram:

- **blue rows** represent the rows returned by an `INNER JOIN` (matching rows from both tables)
- **pink rows** represent additional rows returned by the `LEFT JOIN` — cars that have **no matching owner**


In [25]:
# @title Practice 2
base_practice_2 = make_df_validator_nospoilers(
    expected_hash='4a8a94d3bc19d0f745a83759ebd85b51c0db7f96a7f0f6095fdb91845f52e5c9',
    required_cols=['id', 'name', 'room_id', 'id_1', 'room_number', 'beds', 'floor'],
    expected_rows=15,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_2 = base_practice_2

make_sql_runner(
    conn,
    runner_id="practice_2",
    description_md='### Practice 2\nShow all rows from the table student. If a student is assigned to a room, show the room data as well.\n',
    validator=val_practice_2,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['student', 'room']
)


In [26]:
# @title Practice 3
base_practice_3 = make_df_validator_nospoilers(
    expected_hash='06fe5e8666a7fa5c773f8e0386bfcf70127907c32f40a762974ce2fe8077373a',
    required_cols=['id', 'name', 'room_id', 'id_1', 'room_number', 'beds', 'floor'],
    expected_rows=15,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_3 = base_practice_3

make_sql_runner(
    conn,
    runner_id="practice_3",
    description_md="### Practice 3\nSelect all pieces of equipment together with the room they are assigned to. Show each piece of equipment even if it isn't assigned to a room. Select all available columns.\n",
    validator=val_practice_3,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['equipment', 'room']
)


## RIGHT JOIN explained

Great! As you might expect, there is also a **RIGHT JOIN**.

A `RIGHT JOIN` works similarly to a `LEFT JOIN`, but from the opposite side. It returns:

- **all rows from the right table** (the second table in the query)
- plus any **matching rows from the left table**

If a row from the right table has no matching row in the left table, it will still appear in the result. In that case, the columns from the left table will contain `NULL`.

Let’s look at an example:

```sql
SELECT *
FROM car
RIGHT JOIN person
  ON car.owner_id = person.id;
````

In this query:

* `car` is the **left table**
* `person` is the **right table**

Because we used a `RIGHT JOIN`, SQL will return **all rows from the `person` table**, even if some people do not own a car.

Take a look at the illustration below.

<img src="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/advanced_right_join_example.png" width="60%">

In the diagram:

- **blue rows** represent the result of an `INNER JOIN` (matching rows from both tables)
- **pink rows** represent additional rows returned by the `RIGHT JOIN` — people who **do not own a car**

### Important note

The **order of the tables matters** in `LEFT JOIN` and `RIGHT JOIN`.

For example:

```sql
car RIGHT JOIN person
```

is exactly the same as:

```sql
person LEFT JOIN car
```

So always pay attention to **which table appears first in the query**.


In [27]:
# @title Practice 4
base_practice_4 = make_df_validator_nospoilers(
    expected_hash='7a6894ca402275266fdf6efe0175eebb87ea846676692d64104a0b7486bd56c7',
    required_cols=['id', 'name', 'room_id', 'id_1', 'room_number', 'beds', 'floor'],
    expected_rows=21,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_4 = base_practice_4

make_sql_runner(
    conn,
    runner_id="practice_4",
    description_md='### Practice 4\nFor each student show their data with the data of the room they live in. Show also rooms with no students assigned. Use a RIGHT JOIN.\n',
    validator=val_practice_4,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['student', 'room']
)


In [28]:
# @title Practice 5
base_practice_5 = make_df_validator_nospoilers(
    expected_hash='89c4fa03d8320cca5243d2c42fb11dc2e95fc066e3f184d504b47355a18c4e17',
    required_cols=['id', 'room_number', 'beds', 'floor', 'id_1', 'name', 'room_id'],
    expected_rows=15,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_5 = base_practice_5

make_sql_runner(
    conn,
    runner_id="practice_5",
    description_md='### Practice 5\nFor each student show the room data the student is assigned to. Show also students who are not assigned to any room. Use a RIGHT JOIN.\n',
    validator=val_practice_5,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['student', 'room']
)


## FULL JOIN explained

Excellent! There is one more joining method worth mentioning: **FULL JOIN**.

A `FULL JOIN` returns **all rows from both tables**. Whenever a match is found between the two tables, the rows are combined. If there is no match, the missing side is filled with `NULL` values.

You can think of a `FULL JOIN` as a combination of:
- `LEFT JOIN`
- `RIGHT JOIN`

Let’s look at an example:

```sql
SELECT *
FROM car
FULL JOIN person
  ON car.owner_id = person.id;
````

The result may look like the table below.


<img src="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/advanced_full_join_example.png" width="60%">
          
In the diagram:

- **pink rows** represent the result of an `INNER JOIN` (matching rows from both tables)
- **blue rows** represent rows added by a `LEFT JOIN`
- **purple rows** represent rows added by a `RIGHT JOIN`

A `FULL JOIN` includes **all of them**.

### Note

Unfortunately, not all databases support `FULL JOIN`.


In [29]:
# @title Practice 6
base_practice_6 = make_df_validator_nospoilers(
    expected_hash='6d991b555d69e174cd13be05eaca635fc6d9fc0fc43cce937526fc5d3e183376',
    required_cols=['name', 'room_number'],
    expected_rows=24,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_6 = base_practice_6

make_sql_runner(
    conn,
    runner_id="practice_6",
    description_md="### Practice 6\n\nShow all **rooms and students**, including:  \n\n* rooms that currently have **no students**\n* students who **do not have a room assigned**\n\nYour result should display:\n\n* the student's `name`\n* the `room_number`\n\nUse a **FULL JOIN** between the tables `student` and `room`.\n",
    validator=val_practice_6,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['student', 'room']
)


## The keyword OUTER

Earlier, we mentioned that `JOIN` is actually short for `INNER JOIN`.

A similar shortcut exists for other join types as well.

The joins we discussed earlier:
- `LEFT JOIN`
- `RIGHT JOIN`
- `FULL JOIN`

are technically **outer joins**. Their full names are:

- `LEFT OUTER JOIN`
- `RIGHT OUTER JOIN`
- `FULL OUTER JOIN`

In practice, the word `OUTER` is optional. Adding it does **not change the result** of the query. It simply makes the join type more explicit.

For example, a `LEFT JOIN` can also be written like this:

```sql
SELECT *
FROM person
LEFT OUTER JOIN car
  ON person.id = car.owner_id;
````

Both versions of the query return exactly the same result.


## NATURAL JOIN

There is one more type of join worth mentioning: **NATURAL JOIN**.

It works a little differently from the joins we’ve seen so far because it **does not require an `ON` clause**.

For example:

```sql
SELECT *
FROM person
NATURAL JOIN car;
````

Notice that we didn’t specify any joining condition.

So how does `NATURAL JOIN` work?

Instead of specifying the columns manually, SQL automatically joins the tables using **columns that share the same name in both tables**.

For example, consider this query:

```sql
SELECT *
FROM student
NATURAL JOIN room;
```

In our dataset, both tables contain a column called `id`. Because of this, SQL joins the tables using that column.

In other words, the query behaves like this:

```sql
SELECT *
FROM student
JOIN room
  ON student.id = room.id;
```

In this case, the result doesn’t really make sense, because `student.id` and `room.id` represent completely different things.

---

## When NATURAL JOIN makes sense

`NATURAL JOIN` becomes useful when tables share **meaningful column names**.

For example:

```
car(car_id, brand, model)
owner(owner_id, name, car_id)
```

Here both tables contain the column `car_id`. Using `NATURAL JOIN`, SQL will automatically join the tables on that column:

```sql
SELECT *
FROM car
NATURAL JOIN owner;
```

This can save a few keystrokes because you don’t need to write the `ON` condition explicitly.


## Aliases in self-joins

Great! Aliases are also very useful in another situation: **self-joins**.

Imagine we want to store information about **children and their mothers** in a database. Later, we might want to display each child together with their mother.

Suppose both children and mothers are stored in the same table: `person`.

Each row contains a column called `mother_id`. This column stores the `id` of another row in the same table — the row representing the child's mother.

This raises an interesting question:

**Can we join a table with itself?**

Yes, we can! This is called a **self-join**.

However, we cannot simply write:

```sql
person JOIN person
````

SQL would not know which instance of the table we mean in each part of the query.

To solve this problem, we use **aliases**, temporary names for the same table.

```sql
SELECT *
FROM person child
JOIN person mother
  ON child.mother_id = mother.id;
```

Here we use the table `person` twice:

* `child` represents the rows containing children
* `mother` represents the rows containing mothers

Thanks to these aliases, the database can treat the same table as **two separate tables** within the query and match children with their mothers.


In [30]:
# @title Practice 7
base_practice_7 = make_df_validator_nospoilers(
    expected_hash='4096c4c0f6b28de903da7ec481161a17df7bca25325bbe4201f28bb87aba44d0',
    required_cols=['id', 'name', 'room_id', 'id_1', 'name_1', 'room_id_1'],
    expected_rows=2,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_7 = base_practice_7

make_sql_runner(
    conn,
    runner_id="practice_7",
    description_md='### Practice 7\nWe want to know who lives with the student **Jack Pearson** in the same room. Use self-joining to show all the columns for the student Jack Pearson together with all the columns for each student living with him in the same room.\n',
    validator=val_practice_7,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['student']
)


In [31]:
# @title Practice 8
base_practice_8 = make_df_validator_nospoilers(
    expected_hash='e39b873a3626c533b6045a60d6bfeea3200746effab8dae9bca3b7e5f8956c41',
    required_cols=['name', 'name_1', 'room_number'],
    expected_rows=1,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_8 = base_practice_8

make_sql_runner(
    conn,
    runner_id="practice_8",
    description_md='### Practice 8\nThe challenge is as follows: for each room with 2 beds where there actually are 2 students, we want to show one row which contains the following columns:\n\n  * the name of the first student.\n  * the name of the second student.\n  * the room number.\n\nDon\'t change any column names. Each pair of students should only be shown once. The student whose name comes first in the alphabet should be shown first.\n\nA small hint: in terms of SQL, **"first in the alphabet"** means **"smaller than"** for text values.\n',
    validator=val_practice_8,
    sol_sql=None,
    select_only=True,
    dedupe=True,
    schema_tables=['student', 'room']
)


## CROSS JOIN

Cool! Let’s look at another special type of join: the **CROSS JOIN**.

The general syntax looks like this:

```sql
SELECT *
FROM book
CROSS JOIN author;
````

A `CROSS JOIN` produces what is called a **Cartesian product**.

This means that **every row from the first table is combined with every row from the second table**. In other words, SQL generates **all possible combinations** of rows from the two tables.

For example:

* if the first table has **5 rows**
* and the second table has **3 rows**

the result of the `CROSS JOIN` will contain **15 rows**.

Take a look at the illustration below.

<img src="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/advanced_cross_join_example.png" width="60%">


Each row from the first table is paired with **every row from the second table**, creating all possible row combinations.
